<a href="https://colab.research.google.com/github/ilkecerci/cross-domain-turkish-sentiment-analysis/blob/main/notebooks/01_data_preprocessing_and_split_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import itertools
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Bitirme Projesi/'

# ==========================================
# 1) Product Reviews and Tweets from the raw files
# ==========================================
df_all = pd.concat([pd.read_csv(base_path + 'train.csv'),
                    pd.read_csv(base_path + 'test.csv')], ignore_index=True)

domain_map = {
    'urun_yorumlari': 'Product Reviews',
    'tweet-pn': 'Tweets'
}
others = df_all[df_all['dataset'].isin(domain_map.keys())].copy()
others['Domain'] = others['dataset'].map(domain_map)
others = others.dropna(subset=['text', 'label'])
others = others[others['label'] != 'Notr']
# these domains have no alternative assignment, so approach A/B = original domain
others['domain_keyword'] = others['Domain']
others = others[['text', 'label', 'dataset', 'Domain', 'domain_keyword']]

# ==========================================
# 2) HUMIR (Movie / Hotel) with the original Type labels
# ==========================================
humir = pd.read_csv(base_path + 'humir_domain_labels_AC.csv')
n_humir = len(humir)
humir = humir.dropna(subset=['domain_C']).copy()
print(f"HUMIR rows without original Type (removed): {n_humir - len(humir)}")

humir['Domain'] = humir['domain_C']            # approach C: main domain label
humir['domain_keyword'] = humir['domain_A']    # approach A: kept for comparison
humir['dataset'] = 'HUMIR'
humir = humir[['text', 'label', 'dataset', 'Domain', 'domain_keyword']]

# approach B (ML-based) is added only if the file from the ML step exists
abc_path = base_path + 'humir_domain_labels_ABC.csv'
has_ml = os.path.exists(abc_path)
if has_ml:
    abc = pd.read_csv(abc_path).drop_duplicates(subset='text').set_index('text')
    humir['domain_ml'] = humir['text'].map(abc['domain_B'])
    others['domain_ml'] = others['Domain']
    print(f"domain_ml added, HUMIR rows without B label: {humir['domain_ml'].isna().sum()}")
else:
    print("humir_domain_labels_ABC.csv not found, domain_ml column is skipped for now")

df = pd.concat([others, humir], ignore_index=True)
print(f"\nrows before cleaning: {len(df)}")
print(df['Domain'].value_counts(), "\n")

# ==========================================
# 3) Overlap between domains (before removing duplicates)
# ==========================================
domain_texts = df.groupby('Domain')['text'].apply(set).to_dict()
print("Overlapping samples between domains:")
for a, b in itertools.combinations(sorted(domain_texts), 2):
    print(f"{a} - {b}: {len(domain_texts[a] & domain_texts[b])}")
print()

# ==========================================
# 4) Cleaning: conflicting labels and duplicates
# ==========================================
label_counts = df.groupby('text')['label'].nunique()
conflicting_texts = label_counts[label_counts > 1].index
df = df[~df['text'].isin(conflicting_texts)]
print(f"texts with conflicting labels removed: {len(conflicting_texts)}")

n_dups = df.duplicated(subset='text').sum()
df = df.drop_duplicates(subset='text').reset_index(drop=True)
print(f"duplicate rows removed: {n_dups}")
print(f"rows after cleaning: {len(df)}\n")

assert set(df['label'].unique()) == {'Positive', 'Negative'}

table1 = df.groupby(['Domain', 'label']).size().unstack(fill_value=0)[['Positive', 'Negative']]
table1['Total'] = table1.sum(axis=1)
print("=" * 60)
print("Table 1 - Domain and Label Distribution")
print("=" * 60)
print(table1.to_string(), "\n")

print("Main domain (rows) vs keyword-based domain (columns):")
print(pd.crosstab(df['Domain'], df['domain_keyword']), "\n")

# ==========================================
# 5) Split: stratified by Domain + label (70/10/20)
# ==========================================
df['strat_key'] = df['Domain'] + '_' + df['label']

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['strat_key'])
val_df, test_df = train_test_split(
    temp_df, test_size=2/3, random_state=42, stratify=temp_df['strat_key'])

total = len(df)
for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name}: {len(part)} rows ({len(part) / total:.1%})")
    print(part.groupby('Domain')['label'].value_counts(normalize=True).unstack().round(3))
    print()

# no text should appear in more than one split
assert not set(train_df['text']) & set(val_df['text'])
assert not set(train_df['text']) & set(test_df['text'])
assert not set(val_df['text']) & set(test_df['text'])

cols = [c for c in df.columns if c != 'strat_key']
train_df = train_df[cols]
val_df = val_df[cols]
test_df = test_df[cols]

# ==========================================
# 6) Balanced training set (train only)
# ==========================================
def balance_dataset(data):
    balanced_dfs = []
    for domain in data['Domain'].unique():
        domain_data = data[data['Domain'] == domain]
        min_class_size = domain_data['label'].value_counts().min()

        df_pos = domain_data[domain_data['label'] == 'Positive'].sample(n=min_class_size, random_state=42)
        df_neg = domain_data[domain_data['label'] == 'Negative'].sample(n=min_class_size, random_state=42)

        balanced_dfs.append(pd.concat([df_pos, df_neg]))
    return pd.concat(balanced_dfs).sample(frac=1, random_state=42)

train_balanced = balance_dataset(train_df)
print("train_balanced distribution:")
print(train_balanced.groupby(['Domain', 'label']).size().unstack(fill_value=0), "\n")

# ==========================================
# 7) Save (new _v3 files, old files stay untouched)
# ==========================================
train_df.to_csv(base_path + 'train_4domain_natural_v3.csv', index=False)
val_df.to_csv(base_path + 'val_4domain_natural_v3.csv', index=False)
test_df.to_csv(base_path + 'test_4domain_natural_v3.csv', index=False)
train_balanced.to_csv(base_path + 'train_4domain_balanced_v3.csv', index=False)

print("Saved: train_4domain_natural_v3, val_4domain_natural_v3, test_4domain_natural_v3, train_4domain_balanced_v3")

Mounted at /content/drive
HUMIR rows without original Type (removed): 62
domain_ml added, HUMIR rows without B label: 0

rows before cleaning: 309630
Domain
Product Reviews    234183
Movie Reviews       53230
Hotel Reviews       11162
Tweets              11055
Name: count, dtype: int64 

Overlapping samples between domains:
Hotel Reviews - Movie Reviews: 0
Hotel Reviews - Product Reviews: 2
Hotel Reviews - Tweets: 0
Movie Reviews - Product Reviews: 0
Movie Reviews - Tweets: 0
Product Reviews - Tweets: 2

texts with conflicting labels removed: 76
duplicate rows removed: 3
rows after cleaning: 309474

Table 1 - Domain and Label Distribution
label            Positive  Negative   Total
Domain                                     
Hotel Reviews        5523      5637   11160
Movie Reviews       26586     26644   53230
Product Reviews    219252     14782  234034
Tweets               6084      4966   11050 

Main domain (rows) vs keyword-based domain (columns):
domain_keyword   Hotel Reviews  M

In [ ]:
chk = pd.read_csv(base_path + 'train_4domain_natural_v3.csv')
h = chk[chk['dataset'] == 'HUMIR']
print((h['Domain'] == h['domain_ml']).mean())

0.9894171677057219
